In [ ]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

import nltk
nltk.download("stopwords")
nltk.download("wordnet")

In [ ]:
# Load original dataset
df = pd.read_csv('IMDB.csv')

# Check sentiment distribution
print(df['sentiment'].value_counts())

# Select 250 positive samples
positive_df = df[df['sentiment'] == 'positive'].sample(
    n=250,
    random_state=42
)

# Select 250 negative samples
negative_df = df[df['sentiment'] == 'negative'].sample(
    n=250,
    random_state=42
)

# Combine both datasets
df = pd.concat([positive_df, negative_df])

# Shuffle the final dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save final dataset
df.to_csv('data.csv', index=False)

# Verify distribution
print(df['sentiment'].value_counts())

# Display first rows
df.head()

sentiment
negative    517
positive    483
Name: count, dtype: int64
sentiment
negative    250
positive    250
Name: count, dtype: int64


,review,sentiment
0,This was obviously the worst movie ever made.....,negative
1,Hard to believe this was directed by Fritz Lan...,positive
2,Why is it that a woman cannot be a strong char...,negative
3,Lovingly crafted and terribly interesting to w...,positive
4,Once I watched The Tenant and interpreted it a...,positive


In [45]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [46]:
df = normalize_text(df)
df.head()

,review,sentiment
0,obviously worst movie ever made ketchup starri...,negative
1,hard believe directed fritz lang since mostly ...,positive
2,woman cannot strong character movie without sl...,negative
3,lovingly crafted terribly interesting watch ga...,positive
4,watched tenant interpreted horror movie us man...,positive


In [47]:
df['sentiment'].value_counts()

sentiment
negative    250
positive    250
Name: count, dtype: int64

In [48]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [49]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
0,obviously worst movie ever made ketchup starri...,0
1,hard believe directed fritz lang since mostly ...,1
2,woman cannot strong character movie without sl...,0
3,lovingly crafted terribly interesting watch ga...,1
4,watched tenant interpreted horror movie us man...,1


In [50]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [55]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [53]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/vinaykaladeep/SentimentAnalysisMLOps.mlflow')
dagshub.init(repo_owner='vinaykaladeep', repo_name='SentimentAnalysisMLOps', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as vinaykaladeep

Initialized MLflow to track repo "vinaykaladeep/SentimentAnalysisMLOps"

Repository vinaykaladeep/SentimentAnalysisMLOps initialized!

2026/07/13 19:38:28 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5d705dcdaa9643f2a07c93b95ba52a52', creation_time=1783951709331, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783951709331, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [57]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-13 19:46:09,072 - INFO - Starting MLflow run...
2026-07-13 19:46:10,262 - INFO - Logging preprocessing parameters...
2026-07-13 19:46:11,245 - INFO - Initializing Logistic Regression model...
2026-07-13 19:46:11,247 - INFO - Fitting the model...
2026-07-13 19:46:11,297 - INFO - Model training complete.
2026-07-13 19:46:11,298 - INFO - Logging model parameters...
2026-07-13 19:46:11,640 - INFO - Making predictions...
2026-07-13 19:46:11,645 - INFO - Calculating evaluation metrics...
2026-07-13 19:46:11,686 - INFO - Logging evaluation metrics...
2026-07-13 19:46:12,930 - INFO - Saving and logging the model...
2026/07/13 19:46:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-13 19:46:16,663 - ERROR - An error occurred: cannot import name '_IS_PYPY' from 'sklearn.utils.fixes' (c:\ProgramData\miniconda3\envs\galaxias\lib\site-packages\sklearn\utils\fixes.py)
Traceback (most recent call last):
  File "C:\Users\vinay (vindipop)\AppData\

🏃 View run judicious-dolphin-2 at: https://dagshub.com/vinaykaladeep/SentimentAnalysisMLOps.mlflow/#/experiments/0/runs/9478f677779d4b70afe57b56cf500a31
🧪 View experiment at: https://dagshub.com/vinaykaladeep/SentimentAnalysisMLOps.mlflow/#/experiments/0
